Ingesting posts

In [15]:
import duckdb

# Path to your massive 5.48 GB file
target_file = "/Users/nash/Documents/ConspiracyComments/r_conspiracy_posts.jsonl"

# Query to grab the single absolute maximum timestamp formatted as a readable date
query = f"""
    SELECT 
        MAX(to_timestamp(CAST(created_utc AS BIGINT))) AS latest_post_date
    FROM read_json_auto('{target_file}')
"""

print("Streaming the file to pinpoint the latest timestamp...")
latest_date_df = duckdb.query(query).df()
print("\nExtraction Complete:")
print(latest_date_df)

Streaming the file to pinpoint the latest timestamp...

Extraction Complete:
           latest_post_date
0 2026-03-28 10:19:30+13:00


Checking columns to see if posts contain metadata 

In [23]:
# Check if the scraped metadata already exists in your local data
query = "SELECT url, media, secure_media_embed FROM read_json_auto('/Users/nash/Documents/ConspiracyComments/r_conspiracy_posts.jsonl') WHERE media IS NOT NULL LIMIT 5"
mediadf = duckdb.query(query).df()

In [24]:
mediadf.columns

Index(['url', 'media', 'secure_media_embed'], dtype='object')

In [26]:
for x in mediadf['media']:
    print(x)

{'oembed': {'author_name': 'RT', 'author_url': 'https://www.youtube.com/user/RussiaToday', 'description': "Israel's critics and enemies will see the United States as the side to be blamed for the ongoing violence in the Gaza Strip, believes Congressman Ron Paul. He says the US should review its unconditional support of the Jewish state.", 'height': 338, 'html': '<iframe class="embedly-embed" src="https://cdn.embedly.com/widgets/media.html?src=https%3A%2F%2Fwww.youtube.com%2Fembed%2F6C756NOPyDA%3Ffeature%3Doembed&url=http%3A%2F%2Fwww.youtube.com%2Fwatch%3Fv%3D6C756NOPyDA&image=https%3A%2F%2Fi.ytimg.com%2Fvi%2F6C756NOPyDA%2Fhqdefault.jpg&key=522baf40bd3911e08d854040d3dc5c07&type=text%2Fhtml&schema=youtube" width="600" height="338" scrolling="no" frameborder="0" allowfullscreen></iframe>', 'provider_name': 'YouTube', 'provider_url': 'https://www.youtube.com/', 'thumbnail_height': 90, 'thumbnail_url': 'https://i.embed.ly/1/image?url=https%3A%2F%2Fi.ytimg.com%2Fvi%2F6C756NOPyDA%2Fhqdefault.

In [28]:
for y in mediadf['secure_media_embed']:
    print(y)

{'content': '"<iframe class=\\"embedly-embed\\" src=\\"https://cdn.embedly.com/widgets/media.html?src=https%3A%2F%2Fwww.youtube.com%2Fembed%2F6C756NOPyDA%3Ffeature%3Doembed&url=http%3A%2F%2Fwww.youtube.com%2Fwatch%3Fv%3D6C756NOPyDA&image=https%3A%2F%2Fi.ytimg.com%2Fvi%2F6C756NOPyDA%2Fhqdefault.jpg&key=522baf40bd3911e08d854040d3dc5c07&type=text%2Fhtml&schema=youtube\\" width=\\"600\\" height=\\"338\\" scrolling=\\"no\\" frameborder=\\"0\\" allowfullscreen></iframe>"', 'height': '338', 'media_domain_url': '"https://www.redditmedia.com/mediaembed/7pbs0"', 'scrolling': 'false', 'width': '600'}
{'content': '"<iframe width=\\"459\\" height=\\"344\\" src=\\"https://www.youtube.com/embed/TQ4iIM8Eljc?feature=oembed&enablejsapi=1&enablejsapi=1&enablejsapi=1\\" frameborder=\\"0\\" allow=\\"autoplay; encrypted-media\\" allowfullscreen></iframe>"', 'height': '344', 'media_domain_url': '"https://www.redditmedia.com/mediaembed/7ya99"', 'scrolling': 'false', 'width': '459'}
{'content': '"<iframe width

Getting media titles from posts

In [34]:

import json
import pandas as pd

file_path = '/Users/nash/Documents/ConspiracyComments/r_conspiracy_posts.jsonl'

extracted_records = []
target_limit = 10000

print("Streaming 5.48GB file natively in Python to bypass SQL schema crashes...")

# Open the file and process it line-by-line (highly memory efficient)
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            # Parse the raw string into a Python dictionary
            data = json.loads(line)
            
            # Safely navigate the messy dictionary structure
            media = data.get('media')
            
            # Only proceed if 'media' is actually a dictionary (ignoring False, None, etc.)
            if isinstance(media, dict):
                oembed = media.get('oembed', {})
                
                # Double-check that oembed is also a dictionary
                if isinstance(oembed, dict):
                    title = oembed.get('title')
                    
                    # If we successfully found a title, save the record
                    if title:
                        extracted_records.append({
                            'id': data.get('id', ''),
                            'author': data.get('author', ''),
                            'score': data.get('score', 0),
                            'url': data.get('url', ''),
                            'link_title': title,
                            # Use .get() with a default empty string for missing descriptions
                            'link_description': oembed.get('description') or '' 
                        })
                        
                        # Stop scanning once we hit your limit
                        if len(extracted_records) >= target_limit:
                            break
                            
        except json.JSONDecodeError:
            # If a line is corrupted, just silently skip it
            continue

# Convert our clean list of dictionaries directly into a Pandas DataFrame
links_df = pd.DataFrame(extracted_records)

# Combine the payload for your sentence-transformers model
links_df['combined_payload'] = links_df['link_title'] + " - " + links_df['link_description']

print(f"Extraction successful. Retained {len(links_df)} rich media-link records.")
links_df.head()

Streaming 5.48GB file natively in Python to bypass SQL schema crashes...
Extraction successful. Retained 10000 rich media-link records.


,id,author,score,url,link_title,link_description,combined_payload
0,7pbs0,rhoadesb2,17,http://uk.youtube.com/watch?v=6C756NOPyDA,"Ron Paul on Israel, Crisis, Obama",Israel's critics and enemies will see the Unit...,"Ron Paul on Israel, Crisis, Obama - Israel's c..."
1,7ya99,xandercruise,35,http://www.youtube.com/watch?v=TQ4iIM8Eljc,Cameras in Digital Convert Boxes! BEWARE!!!!,,Cameras in Digital Convert Boxes! BEWARE!!!! -
2,90dai,CurtisDuncan,3,http://www.youtube.com/watch?v=6hpPkLKucaA,"Killer Swine Flu Vaccine Soon to Be Released, ...",,"Killer Swine Flu Vaccine Soon to Be Released, ..."
3,9ips8,plutocrat24,1,http://www.youtube.com/watch?v=Tt7zD9p80gA,Eye In The Sky,,Eye In The Sky -
4,9uv2x,antifacist,0,http://www.ustream.tv/recorded/1391917,The Great Global Warming Swindle,Global Warming Propaganda Exposed,The Great Global Warming Swindle - Global Warm...


Exploratory analysis of comments

In [6]:
import pandas as pd

# Define your file path
file_path = '/Users/nash/Documents/ConspiracyComments/r_conspiracy_comments3.jsonl'

# Load just the first 10,000 rows to explore
df = pd.read_json(file_path, nrows=10000, lines = True)

# Display the first 5 rows to see what the columns look like
df.head()

,all_awardings,associated_award,author,author_created_utc,author_flair_background_color,author_flair_css_class,author_flair_richtext,author_flair_template_id,author_flair_text,author_flair_text_color,...,subreddit,subreddit_id,subreddit_name_prefixed,subreddit_type,top_awarded_type,total_awards_received,treatment_tags,name,ups,author_cakeday
0,[],NaN,LurkingFromBelow,NaN,None,None,[],NaN,None,None,...,conspiracy,t5_2qh4r,r/conspiracy,public,NaN,0,[],t1_gfz82vv,3,NaN
1,[],NaN,redditready1986,1.337874e+09,None,None,[],NaN,None,None,...,conspiracy,t5_2qh4r,r/conspiracy,public,NaN,0,[],t1_gfz82x0,0,NaN
2,[],NaN,bgny,1.291781e+09,None,None,[],NaN,None,None,...,conspiracy,t5_2qh4r,r/conspiracy,public,NaN,0,[],t1_gfz830e,7,NaN
3,[],NaN,walkclothed,1.348939e+09,None,None,[],NaN,None,None,...,conspiracy,t5_2qh4r,r/conspiracy,public,NaN,0,[],t1_gfz83rj,2,NaN
4,[],NaN,Myskinisnotmyown,1.370322e+09,None,None,[],NaN,None,None,...,conspiracy,t5_2qh4r,r/conspiracy,public,NaN,0,[],t1_gfz83sl,2,NaN


In [2]:
df.columns

Index(['all_awardings', 'associated_award', 'author', 'author_created_utc',
       'author_flair_background_color', 'author_flair_css_class',
       'author_flair_richtext', 'author_flair_template_id',
       'author_flair_text', 'author_flair_text_color', 'author_flair_type',
       'author_fullname', 'author_patreon_flair', 'author_premium', 'awarders',
       'body', 'can_gild', 'can_mod_post', 'collapsed',
       'collapsed_because_crowd_control', 'collapsed_reason', 'comment_type',
       'controversiality', 'created_utc', 'distinguished', 'edited', 'gilded',
       'gildings', 'id', 'is_submitter', 'link_id', 'locked', 'no_follow',
       'parent_id', 'permalink', 'quarantined', 'removal_reason',
       'retrieved_on', 'score', 'send_replies', 'stickied', 'subreddit',
       'subreddit_id', 'subreddit_name_prefixed', 'subreddit_type',
       'top_awarded_type', 'total_awards_received', 'treatment_tags', 'name',
       'ups', 'author_cakeday'],
      dtype='object')

In [4]:
for comment in df['body'][:10]:
    print(comment)

I think owls eat mice entirely and then spit up balls of hair and fur later. Im not certain, i havnt seen much animal related stuff recently. 


Fun story though, fits the sub.
It's the least I could do. I am all about helping folks that have disabilities.
It wasn't good that no one listened to him, because he was right.  Guess you've been brainwashed away from the truth by communists, because you don't do your own research in to the subject.
Fill that Condom up with silicone lube and insert a bullet vibrator so it sits right under the head and thank me in the morning

/probator
No one said it was, friend.
i dont think anyone will disagree
You're playing with fire emboldening dark forces. Who follows you to recover the funds using blackmail?
> Patriots overrode the crown, then the south.

You know, the south considered themselves "patriots" too, and that they were fighting for some version of "TRUTH" and against "TYRANNY". They convinced themselves that a presidential election was ille

Incorporating comments in duckdb

In [7]:
import duckdb
import pandas as pd

# 1. Define your files. You can type them out as a list, 
# or if they are all in a folder, just use a wildcard like 'downloads/*.jsonl'
file_paths = ['/Users/nash/Documents/ConspiracyComments/*.jsonl']

# 2. Write the DuckDB Query with a deduplication step
query = f"""
    -- Step A: Combine all files and drop the overlapping duplicates based on the unique comment 'id'
    WITH deduplicated_comments AS (
        SELECT DISTINCT id, parent_id
        FROM read_json_auto({file_paths})
    )
    
    -- Step B: Run our reply-counting math on the clean, deduplicated list
    SELECT 
        parent_id, 
        COUNT(*) as reply_count
    FROM deduplicated_comments
    WHERE parent_id LIKE 't1_%'
    GROUP BY parent_id
    ORDER BY reply_count DESC
    LIMIT 500
"""

# 3. Execute
print("Scanning multiple files, filtering out overlaps, and crunching the numbers...")
battlegrounds_df = duckdb.query(query).df()

print("Scan complete. Here are your top 5 battlegrounds:")
battlegrounds_df.head()

Scanning multiple files, filtering out overlaps, and crunching the numbers...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scan complete. Here are your top 5 battlegrounds:


,parent_id,reply_count
0,t1_g0ylmj4,1723
1,t1_dka1xu5,797
2,t1_g0iaeb4,293
3,t1_n6f2gh1,222
4,t1_g7eej7n,212


In [6]:
battlegrounds_df

,parent_id,reply_count
0,t1_g0ylmj4,1723
1,t1_dka1xu5,797
2,t1_g0iaeb4,293
3,t1_n6f2gh1,222
4,t1_g7eej7n,212
...,...,...
495,t1_k5ptlai,49
496,t1_j8hi7qs,49
497,t1_gki03fy,49
498,t1_diy76g5,49


In [8]:
import duckdb
import pandas as pd
import re

# 1. Your files and your target ID
file_paths = ['/Users/nash/Documents/ConspiracyComments/*.jsonl'] # Update this to your list
target_id = 'g0iaeb4' # Replace with your top ID (without the 't1_' prefix)


# Clean the target_id in Python just to be safe (removes t1_ or t3_ if you accidentally pasted it)
clean_target = target_id.replace('t1_', '').replace('t3_', '')

# The Bulletproof Query using wildcards (%)
query = f"""
    SELECT id, parent_id, author, score, body
    FROM read_json_auto({file_paths})
    WHERE id LIKE '%{clean_target}' 
       OR parent_id LIKE '%{clean_target}'
"""

print(f"Extracting thread for {clean_target} using wildcards...")
thread_df = duckdb.query(query).df()

# Split them up again (using string containment to be safe)
parent_comment = thread_df[thread_df['id'].str.contains(clean_target, na=False)]
replies = thread_df[thread_df['parent_id'].str.contains(clean_target, na=False)]

print(f"Extraction complete. Found the parent comment and {len(replies)} direct replies.")
# 2. The Extraction Query
# We are asking DuckDB to grab the parent comment itself (id = target) 
# AND every comment that directly replied to it (parent_id = t1_target)
query = f"""
    SELECT id, parent_id, author, score, body
    FROM read_json_auto({file_paths})
    WHERE id = '{target_id}' OR parent_id = 't1_{target_id}'
"""

print(f"Extracting the conversation thread for {target_id}...")
thread_df = duckdb.query(query).df()

# 3. Split the DataFrame into 'The Claim' and 'The Debate'
parent_comment = thread_df[thread_df['id'] == target_id]
replies = thread_df[thread_df['parent_id'] == f't1_{target_id}']

print(f"Extraction complete. Found the parent comment and {len(replies)} direct replies.")

Extracting thread for g0iaeb4 using wildcards...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extraction complete. Found the parent comment and 293 direct replies.
Extracting the conversation thread for g0iaeb4...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extraction complete. Found the parent comment and 293 direct replies.


In [9]:
# 1. Create a new column containing the length of the string
# We use fillna('') just in case there are any empty rows so it doesn't throw an error
replies['body_length'] = replies['body'].fillna('').str.len()

# 2. Sort the replies from longest to shortest
longest_replies = replies.sort_values(by='body_length', ascending=False)

# 3. Print the top 5 longest essays
print("THE 5 LONGEST REPLIES IN THE THREAD")
print("=========================================")

for index, row in longest_replies.head(5).iterrows():
    print(f"\nAuthor: {row['author']} | Character Count: {row['body_length']}")
    print("-" * 40)
    print(row['body'])
    print("\n" + "*"*40)

THE 5 LONGEST REPLIES IN THE THREAD

Author: Donald_John-Trump | Character Count: 4801
----------------------------------------
[1](https://np.reddit.com/r/conspiracy/comments/i30m7u/trump_said_this_in_2012_now_hes_president/)

[2](https://np.reddit.com/r/conspiracy/comments/ht3blh/trump_warns_epsteins_island_a_cesspool_in_2015/)

[3](https://np.reddit.com/r/conspiracy/comments/hxxfsp/black_trump_supporter_shot_dead_after_interview/)

[4](https://np.reddit.com/r/conspiracy/comments/fqxjt9/picture_on_rpics_comparing_xi_to_the_coronavirus/)

[5](https://np.reddit.com/r/conspiracy/comments/i1r1g1/bill_clinton_named_on_pedo_island_in_actually/)

[6](https://np.reddit.com/r/conspiracy/comments/g3cnfo/70k_upvotes_on_a_post_claiming_trump_supporter/)

[7](https://np.reddit.com/r/conspiracy/comments/fgqu21/based_plant_lady_knows_joe_bidens_campaign_is_a/)

Or how about the numerous comments from pro Trump accounts that got approved despite breaking rules?

[1](https://np.reddit.com/r/conspirac

/var/folders/ym/6mr3ypl97rj6468b4m65lq240000gp/T/ipykernel_58834/3519156370.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  replies['body_length'] = replies['body'].fillna('').str.len()


In [23]:
longest_replies

,id,parent_id,author,score,body,body_length
11,g0ih5ig,t1_g0iaeb4,Donald_John-Trump,2474.0,[1](https://np.reddit.com/r/conspiracy/comment...,4801
75,g0j7a7f,t1_g0iaeb4,Distasteful_Username,880.0,The person who posted all those links is cool ...,3787
282,g0m8dzt,t1_g0iaeb4,Alles_Spice,2.0,I mean no disrespect to the mods but I think y...,2578
83,g0j956r,t1_g0iaeb4,judgecucken72,57.0,[This](https://www.reddit.com/r/conspiracy/com...,2150
86,g0jay9q,t1_g0iaeb4,OracularLettuce,77.0,"So for starters, if you ""*see posts like yours...",1835
...,...,...,...,...,...,...
128,g0jxzzj,t1_g0iaeb4,Gigatron_0,4.0,Trash,5
178,g0kmqu5,t1_g0iaeb4,yaboimissesezlayups,3.0,Rekt,4
107,g0jkp9k,t1_g0iaeb4,TOADSTOOL__SURPRISE,1.0,Lol,3
31,g0isnfd,t1_g0iaeb4,Cocoadicks,2.0,Lol,3


In [10]:
battlegrounds_df.columns

Index(['parent_id', 'reply_count'], dtype='object')

In [11]:
import duckdb

# Point this to your folder of JSONL files
file_paths = ['/Users/nash/Documents/ConspiracyComments/*.jsonl']

query = f"""
    SELECT 
        MIN(to_timestamp(CAST(created_utc AS BIGINT))) AS earliest_comment,
        MAX(to_timestamp(CAST(created_utc AS BIGINT))) AS most_recent_comment
    FROM read_json_auto({file_paths})
"""

print("Scanning files for the timeline boundaries...")
date_range_df = duckdb.query(query).df()
date_range_df

Scanning files for the timeline boundaries...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_comment,most_recent_comment
0,2008-01-30 03:02:37+13:00,2026-03-31 11:11:16+13:00


setting up zero shot embeddings for search of comments to identify disussions around epistemic credibility

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer, util

# 1. Load a lightweight, highly accurate embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Define your precise semantic anchors (keep them lean and conceptual)
anchors = [
    "There is overwhelming empirical evidence to support this.",
    "This is the established consensus among field experts.",
    "The data definitively proves this assertion.",
    "There is zero verifiable evidence for that claim.",
    "That assertion is clearly false and unsupported.",
    "You cannot reasonably believe that based on current data."
]

# 3. Pre-compute anchor embeddings (do this once and cache them)
anchor_embeddings = model.encode(anchors, convert_to_tensor=True)

def analyze_text(input_text, threshold=0.45):
    """
    Checks if the input text semantically aligns with any of your core concepts.
    """
    # Embed the incoming target text
    target_embedding = model.encode(input_text, convert_to_tensor=True)
    
    # Compute cosine similarities between target and all anchors
    similarities = util.cos_sim(target_embedding, anchor_embeddings)[0]
    
    # Find the best matching anchor
    max_score_idx = np.argmax(similarities.cpu().numpy())
    max_score = similarities[max_score_idx].item()
    best_match = anchors[max_score_idx]
    
    # Print diagnostic evaluation
    print(f"Input: '{input_text}'")
    print(f"Best Anchor Match: '{best_match}'")
    print(f"Semantic Match Score: {max_score:.4f}")
    
    if max_score >= threshold:
        print("Verdict: Strong conceptual alignment detected.\n")
        return True
    else:
        print("Verdict: Below threshold. Ignored.\n")
        return False

# --- Quick Test Loop ---
print("--- Running Concept Vector Verification ---\n")
analyze_text("The peer-reviewed literature shows a clear correlation.")
analyze_text("I reckon it's true because my uncle told me so on the weekend.")

README.md: 0.00B [00:00, ?B/s]

--- Running Concept Vector Verification ---

Input: 'The peer-reviewed literature shows a clear correlation.'
Best Anchor Match: 'There is overwhelming empirical evidence to support this.'
Semantic Match Score: 0.4043
Verdict: Below threshold. Ignored.

Input: 'I reckon it's true because my uncle told me so on the weekend.'
Best Anchor Match: 'That assertion is clearly false and unsupported.'
Semantic Match Score: 0.3026
Verdict: Below threshold. Ignored.



False

Anchors based on different epistemic stances

In [13]:
anchors_absolutist = ["This is an undisputed fact.", "The data proves this absolutely."]
anchors_evaluativist = ["The balance of evidence supports this interpretation.", "We must weigh the counterarguments."]
anchors_hedges = ["It could be argued that...", "Perhaps there is a correlation."]

Loading factappeal dataset 

https://arxiv.org/abs/2510.10658

Retrieved from https://github.com/guymorlan/factappeal.git

In [35]:
import pandas as pd

# Point Pandas to the new folder in your home directory
local_train_path = '/Users/nash/factappeal/train.csv'

print("Loading local FACTAPPEAL dataset...")
fact_df = pd.read_csv(local_train_path)

print(f"Loaded {len(fact_df)} annotated sentences.")
fact_df.head()

Loading local FACTAPPEAL dataset...
Loaded 2256 annotated sentences.


,sentence,annotation
0,Using campaign funds for personal use is a vio...,<Fact_No_Appeal>Using campaign funds for perso...
1,The move by North Korea was the first of its k...,<Fact_No_Appeal>The move by North Korea was th...
2,It seems like her job had an insurance policy ...,<Fact_No_Appeal>It seems like her job had an i...
3,He knew that his own parents had been investig...,<Fact_No_Appeal>He knew that his own parents h...
4,What was also interesting is that the shy ones...,<Fact_No_Appeal>What was also interesting is t...


In [36]:
import duckdb

con = duckdb.connect()
output_file = '/Users/nash/Documents/ConspiracyComments/lexical_scores_full.parquet'

# Extract all URLs and their domains, with upvotes
print(con.execute(f"""
    WITH urls AS (
        SELECT 
            id,
            upvotes,
            controversiality,
            -- Extract the full URL
            UNNEST(regexp_extract_all(text, 'https?://[^\s\)\]"]+')) AS url
        FROM '{output_file}'
        WHERE text LIKE '%http%'
    ),
    domains AS (
        SELECT
            id,
            upvotes,
            controversiality,
            url,
            -- Extract just the domain
            regexp_extract(url, 'https?://(?:www\.)?([^/\s]+)') AS domain
        FROM urls
    )
    SELECT 
        domain,
        COUNT(*) as citations,
        ROUND(AVG(upvotes), 2) as avg_upvotes,
        ROUND(MEDIAN(upvotes), 2) as median_upvotes,
        ROUND(AVG(controversiality), 3) as avg_controversy,
        COUNT(DISTINCT id) as unique_comments
    FROM domains
    WHERE domain != ''
      AND length(domain) > 3
    GROUP BY domain
    ORDER BY citations DESC
    LIMIT 50
""").df().to_string())

<>:48: SyntaxWarning: invalid escape sequence '\s'
<>:51: SyntaxWarning: invalid escape sequence '\.'
<>:48: SyntaxWarning: invalid escape sequence '\s'
<>:51: SyntaxWarning: invalid escape sequence '\.'
/var/folders/ym/6mr3ypl97rj6468b4m65lq240000gp/T/ipykernel_58834/4216252512.py:48: SyntaxWarning: invalid escape sequence '\s'
/var/folders/ym/6mr3ypl97rj6468b4m65lq240000gp/T/ipykernel_58834/4216252512.py:51: SyntaxWarning: invalid escape sequence '\.'


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                             domain  citations  avg_upvotes  median_upvotes  avg_controversy  unique_comments
0            https://www.reddit.com    2475946         1.32             1.0            0.005          1298010
1           https://www.youtube.com     245849         5.54             2.0            0.065           158502
2          https://en.wikipedia.org     153453         5.92             2.0            0.062           101099
3                  https://youtu.be     149249         5.51             2.0            0.068           108728
4             https://np.reddit.com     115107         4.09             1.0            0.024            48568
5               https://twitter.com      58882         8.87             2.0            0.131            41644
6                https://archive.is      58856         5.27             1.0            0.050            49655
7               https://i.imgur.com      48922         6.44             1.0            0.077            27549
8        h

In [37]:
import duckdb
con = duckdb.connect()

# Just look at all the columns available in one file
result = con.execute("""
    SELECT * 
    FROM read_json_auto('/Users/nash/Documents/ConspiracyComments/r_conspiracy_comments3.jsonl',
                        ignore_errors=true)
    LIMIT 1
""").df()

print(result.columns.tolist())

['all_awardings', 'associated_award', 'author', 'author_created_utc', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'author_fullname', 'author_patreon_flair', 'author_premium', 'awarders', 'body', 'can_gild', 'can_mod_post', 'collapsed', 'collapsed_because_crowd_control', 'collapsed_reason', 'comment_type', 'controversiality', 'created_utc', 'distinguished', 'edited', 'gilded', 'gildings', 'id', 'is_submitter', 'link_id', 'locked', 'no_follow', 'parent_id', 'permalink', 'quarantined', 'removal_reason', 'retrieved_on', 'score', 'send_replies', 'stickied', 'subreddit', 'subreddit_id', 'subreddit_name_prefixed', 'subreddit_type', 'top_awarded_type', 'total_awards_received', 'treatment_tags', 'name', 'ups', 'author_cakeday']


In [38]:
import duckdb
import time

con = duckdb.connect()
PARQUET_PATH = '/Users/nash/Documents/ConspiracyComments/lexical_scores_full.parquet'

print("Initializing tractable deep-path extractions...")
start_time = time.time()

# 1. WIKIPEDIA HISTORICAL CANON
print("Scanning Wikipedia paths...")
con.execute(f"""
    COPY (
        WITH raw_urls AS (
            SELECT unnest(regexp_extract_all(text, 'https?://en\\\\.wikipedia\\\\.org/wiki/[^\\\\s\\\\)\\\\\\\\]\\"]+')) AS url
            FROM '{PARQUET_PATH}'
            WHERE text LIKE '%wikipedia.org/wiki/%'
        ),
        cleaned_paths AS (
            SELECT regexp_extract(url, '/wiki/([^\\\\s\\\\)\\\\]\\"]+)', 1) AS wiki_slug
            FROM raw_urls
        )
        SELECT wiki_slug, COUNT(*) as reference_count
        FROM cleaned_paths
        WHERE wiki_slug != ''
        GROUP BY wiki_slug
        ORDER BY reference_count DESC
        LIMIT 100
    ) TO '/Users/nash/Documents/ConspiracyComments/top_wikipedia_canon.csv' (FORMAT CSV, HEADER);
""")

# 2. YOUTUBE CANON
print("Scanning YouTube Video IDs...")
con.execute(f"""
    COPY (
        WITH raw_urls AS (
            SELECT unnest(regexp_extract_all(text, 'https?://(?:www\\\\.)?(?:youtube\\\\.com|youtu\\\\.be)/[^\\\\s\\\\)\\\\\\\\]\\"]+')) AS url
            FROM '{PARQUET_PATH}'
            WHERE text LIKE '%youtu%'
        ),
        video_ids AS (
            SELECT regexp_extract(url, '(?:v=|\\\\/v\\\\/|embed\\\\/|youtu\\\\.be\\\\/)([^?&\\\\s\\\\)\\\\]\\"]+)', 1) AS video_id
            FROM raw_urls
        )
        SELECT video_id, COUNT(*) as reference_count
        FROM video_ids
        WHERE length(video_id) = 11
        GROUP BY video_id
        ORDER BY reference_count DESC
        LIMIT 100
    ) TO '/Users/nash/Documents/ConspiracyComments/top_youtube_videos.csv' (FORMAT CSV, HEADER);
""")

# 3. PUBMED WEAPONIZED SCIENCE
print("Scanning PubMed identifiers...")
con.execute(f"""
    COPY (
        WITH raw_urls AS (
            SELECT unnest(regexp_extract_all(text, 'https?://(?:www\\\\.)?ncbi\\\\.nlm\\\\.nih\\\\.gov/pubmed/[0-9]+')) AS url
            FROM '{PARQUET_PATH}'
            WHERE text LIKE '%ncbi.nlm.nih.gov/pubmed%'
        ),
        pubmed_ids AS (
            SELECT regexp_extract(url, '/pubmed/([0-9]+)', 1) AS pm_id
            FROM raw_urls
        )
        SELECT pm_id, COUNT(*) as reference_count
        FROM pubmed_ids
        GROUP BY pm_id
        ORDER BY reference_count DESC
        LIMIT 100
    ) TO '/Users/nash/Documents/ConspiracyComments/top_pubmed_studies.csv' (FORMAT CSV, HEADER);
""")

elapsed = time.time() - start_time
print(f"Extraction complete in {elapsed:.1f} seconds! Files saved to your project directory.")

Initializing tractable deep-path extractions...
Scanning Wikipedia paths...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scanning YouTube Video IDs...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Scanning PubMed identifiers...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Extraction complete in 10.5 seconds! Files saved to your project directory.


In [40]:
import duckdb
con = duckdb.connect()
PARQUET_PATH = '/Users/nash/Documents/ConspiracyComments/lexical_scores_full.parquet'

# Wikipedia slugs
print(con.execute(f"""
    SELECT 
        regexp_extract(url, '/wiki/([^\\s\\)\\]"]+)') AS wiki_slug,
        COUNT(*) as n
    FROM (
        SELECT UNNEST(regexp_extract_all(text, 'wikipedia.org/wiki/[^\\s\\)\\]"]+')) AS url
        FROM '{PARQUET_PATH}'
        WHERE text LIKE '%wikipedia.org/wiki/%'
    )
    WHERE wiki_slug != ''
    GROUP BY wiki_slug
    ORDER BY n DESC
    LIMIT 30
""").df().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                                     wiki_slug     n
0                                     /wiki/Stockholm_syndrome  1733
1                                  /wiki/Operation_Mockingbird   691
2                                   /wiki/Operation_Northwoods   533
3                                      /wiki/Haavara_Agreement   405
4                                            /wiki/Consumerism   360
5                                /wiki/New_York_Stock_Exchange   347
6                                         /wiki/Reality_tunnel   334
7                                             /wiki/COINTELPRO   313
8                                    /wiki/Operation_Paperclip   299
9             /wiki/Donald_Trump_sexual_misconduct_allegations   296
10  /wiki/Unethical_human_experimentation_in_the_United_States   289
11                            /wiki/Foundations_of_Geopolitics   285
12                  /wiki/Project_for_the_New_American_Century   284
13                             /wi

In [39]:
result2 = con.execute("""
    SELECT * 
    FROM read_json_auto('/Users/nash/Documents/ConspiracyComments/r_conspiracy_posts.jsonl',
                        ignore_errors=true)
    LIMIT 1
""").df()

print(result2.columns.tolist())

['archived', 'author', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'brand_safe', 'can_gild', 'contest_mode', 'created_utc', 'distinguished', 'domain', 'edited', 'gilded', 'hidden', 'hide_score', 'id', 'is_crosspostable', 'is_reddit_media_domain', 'is_self', 'is_video', 'link_flair_css_class', 'link_flair_richtext', 'link_flair_text', 'link_flair_text_color', 'link_flair_type', 'locked', 'media', 'media_embed', 'no_follow', 'num_comments', 'num_crossposts', 'over_18', 'parent_whitelist_status', 'permalink', 'retrieved_on', 'rte_mode', 'score', 'secure_media', 'secure_media_embed', 'selftext', 'send_replies', 'spoiler', 'stickied', 'subreddit', 'subreddit_id', 'subreddit_name_prefixed', 'subreddit_type', 'suggested_sort', 'thumbnail', 'thumbnail_height', 'thumbnail_width', 'title', 'url', 'whitelist_status', 'name', 'ups', 'upvote_ratio', 'author_cakeday', 'post_hint', 'preview', 

In [41]:
import duckdb
con = duckdb.connect()
PARQUET_PATH = '/Users/nash/Documents/ConspiracyComments/lexical_scores_full.parquet'

print(con.execute(f"""
    SELECT 
        regexp_extract(url, '(?:v=|/v/|embed/|youtu\\.be/)([^?&\\s\\)\\]"]+)', 1) AS video_id,
        COUNT(*) as n
    FROM (
        SELECT UNNEST(regexp_extract_all(text, 'https?://(?:www\\.)?(?:youtube\\.com|youtu\\.be)/[^\\s\\)\\]"]+')) AS url
        FROM '{PARQUET_PATH}'
        WHERE text LIKE '%youtu%'
    )
    WHERE length(video_id) = 11
    GROUP BY video_id
    ORDER BY n DESC
    LIMIT 30
""").df().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       video_id     n
0   rdrKCilEhC0  1282
1   O1GCeuSr3Mk   520
2   7Eeo-82Eac8   469
3   bXmuWecIQos   356
4   KpuKu3F0BvY   349
5   6_s1Zbs20aA   347
6   U1Qt6a-vaNM   311
7   Yqb59n69Z80   282
8   uW3a1bw5XlE   274
9   dQw4w9WgXcQ   271
10  09maaUaRT4M   269
11  yuC_4mGTs98   267
12  8DOnAn_PX6M   251
13  9RC1Mepk_Sw   250
14  t_fOYW-GXdc   248
15  gbUK4cFCTPg   237
16  Ddz2mw2vaEg   232
17  KYL94OjXHYY   225
18  5EarOB411BA   224
19  rf78rEAJvhY   221
20  -bYAQ-ZZtEU   219
21  e6Yif7LGFCc   208
22  eJ3RzGoQC4s   207
23  G45WthPTo24   203
24  55tdrnP4rxc   202
25  JOktR-FbzvU   201
26  KVeDKuHPDK8   198
27  ZtlhThIk3iI   197
28  yCzdliixnmI   196
29  -GZFHLAcG8A   191


In [42]:
print(con.execute(f"""
    SELECT 
        regexp_extract(url, '/pubmed/([0-9]+)', 1) AS pubmed_id,
        COUNT(*) as n
    FROM (
        SELECT UNNEST(regexp_extract_all(text, 'https?://(?:www\\.)?ncbi\\.nlm\\.nih\\.gov/pubmed/[0-9]+')) AS url
        FROM '{PARQUET_PATH}'
        WHERE text LIKE '%pubmed%'
    )
    WHERE pubmed_id != ''
    GROUP BY pubmed_id
    ORDER BY n DESC
    LIMIT 30
""").df().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   pubmed_id    n
0   12145534  195
1   21623535  187
2   17454560  164
3   21058170  151
4   21993250  141
5   11339848  130
6   17114826  122
7   12933322  107
8   22099159  105
9   12849883  102
10  22591873   99
11  21299355   98
12  19106436   96
13  25377033   96
14  19043938   95
15  23902317   95
16  15780490   94
17  24675092   94
18  16870260   93
19  17674242   91
20  12142947   91
21  21907498   89
22  24995277   84
23  18445737   67
24  25198681   51
25  18454680   49
26  23932735   46
27  24791031   45
28  17952650   44
29  19216002   41


In [43]:
import requests
import time

pubmed_ids = ['12145534', '21623535', '17454560', '21058170', '21993250', 
              '11339848', '17114826', '12933322', '22099159', '12849883',
              '22591873', '21299355', '19106436', '25377033', '19043938']

print("Fetching PubMed titles...")
for pmid in pubmed_ids:
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=pubmed&id={pmid}&retmode=json"
    r = requests.get(url)
    data = r.json()
    try:
        title = data['result'][pmid]['title']
        source = data['result'][pmid]['source']
        print(f"{pmid}: {title} [{source}]")
    except:
        print(f"{pmid}: lookup failed")
    time.sleep(0.4)  # be polite to NCBI

Fetching PubMed titles...
12145534: Abnormal measles-mumps-rubella antibodies and CNS autoimmunity in children with autism. [J Biomed Sci]
21623535: A positive association found between autism prevalence and childhood vaccination uptake across the U.S. population. [J Toxicol Environ Health A]
17454560: A case series of children with apparent mercury toxic encephalopathies manifesting with clinical symptoms of regressive autistic disorders. [J Toxicol Environ Health A]
21058170: Hepatitis B vaccination of male neonates and autism diagnosis, NHIS 1997-2002. [J Toxicol Environ Health A]
21993250: Hypothesis: conjugate vaccines may predispose children to autism spectrum disorders. [Med Hypotheses]
11339848: Autism: a novel form of mercury poisoning. [Med Hypotheses]
17114826: Aluminum adjuvant linked to Gulf War illness induces motor neuron death in mice. [Neuromolecular Med]
12933322: Reduced levels of mercury in first baby haircuts of autistic children. [Int J Toxicol]
22099159: Do alumi

In [44]:
import duckdb

con = duckdb.connect()

print("Scanning submission metadata...")
query = """
SELECT 
    domain,
    COUNT(*) as total_submissions,
    ROUND(AVG(score), 2) as avg_score,
    MAX(score) as top_score
FROM read_json_auto('/Users/nash/Documents/ConspiracyComments/r_conspiracy_posts*.jsonl', union_by_name=true, ignore_errors=true)
WHERE domain IS NOT NULL 
  AND domain != 'self.conspiracy'
  AND domain != ''
GROUP BY domain
ORDER BY total_submissions DESC
LIMIT 30
"""

print(con.execute(query).df().to_string())

Scanning submission metadata...
                 domain  total_submissions  avg_score  top_score
0             i.redd.it             197111     206.08      76121
1           youtube.com             148570      27.58      64503
2              youtu.be              88698      23.21      30419
3            reddit.com              33893      64.72      33132
4           twitter.com              23960     111.39      31560
5           i.imgur.com              19167     268.15      36780
6             imgur.com              16803     113.09      26225
7         zerohedge.com              10105      56.58      10758
8         m.youtube.com               8787      27.74       7457
9                rt.com               8031      41.02       6714
10      dailymail.co.uk               7690      86.79      17006
11      theguardian.com               7536      48.62       7986
12     activistpost.com               5637      39.45       6879
13          nytimes.com               5632      34.67     

In [45]:
import duckdb

con = duckdb.connect()

print("Extracting top specific URLs and their submission titles...")
query = """
SELECT 
    url,
    -- We take the most commonly used title for this specific URL
    MODE(title) as most_common_title,
    COUNT(*) as submission_count,
    ROUND(AVG(score), 2) as avg_score
FROM read_json_auto('/Users/nash/Documents/ConspiracyComments/r_conspiracy_posts*.jsonl', union_by_name=true, ignore_errors=true)
WHERE url IS NOT NULL 
  AND domain NOT IN ('i.redd.it', 'v.redd.it', 'i.imgur.com', 'imgur.com', 'reddit.com', 'np.reddit.com') -- Filter out pure image/internal links
  AND domain != 'self.conspiracy'
GROUP BY url
HAVING submission_count > 10 -- Only show URLs that have been submitted multiple times
ORDER BY submission_count DESC
LIMIT 30
"""

print(con.execute(query).df().to_string(max_colwidth=80))

Extracting top specific URLs and their submission titles...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                                                                url                                                                most_common_title  submission_count  avg_score
0                                                                                                                                                  [deleted by user]             29429      12.98
1   https://www.google.com/amp/s/abc7news.com/amp/sources-jeffrey-epstein-dies-b...                  Sources: Jeffrey Epstein dies by suicide in Manhattan jail cell               121     268.54
2              https://youtube.com/playlist?list=PLmUOtaC-dPk0gGZqxlKzkXCG3Y7m9rUNL                                                                  Occult classics                79       1.25
3                                   http://theinversionofchristianity.blogspot.com/                                                  My views on the New World Order                77       0.82
4                             

In [46]:
query = """
SELECT 
    title,
    domain,
    score,
    num_comments,
    upvote_ratio,
    created_utc
FROM read_json_auto('/Users/nash/Documents/ConspiracyComments/r_conspiracy_posts*.jsonl', 
                    union_by_name=true, ignore_errors=true)
WHERE score > 1000
  AND title IS NOT NULL
  AND domain NOT IN ('i.redd.it', 'v.redd.it', 'i.imgur.com', 'imgur.com')
ORDER BY score DESC
LIMIT 50
"""
print(con.execute(query).df()[['title','domain','score','num_comments','upvote_ratio']].to_string())

                                                                                                                                                                                                                                                                                title                    domain  score  num_comments  upvote_ratio
0                                                                                                                                                                                                                                    This was deleted twice from reddit's front page.               youtube.com  64503          2175          1.00
1                                                                                                                                 REMINDER: It has been 2866 days since Sean Hannity offered to undergo waterboarding for charity as proof that it's not torture and has not done it.        crooksandliars.com  42498          14